# EDA 3 — So sánh 2 bộ và gợi ý tiền xử lý
Đọc `reports/summary_*.json` do 2 notebook trước sinh ra (chạy 01–02 trước).

In [1]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from eda_utils import *
warnings.filterwarnings('ignore'); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 80
RAW = ROOT / 'data' / 'raw'
import json

In [2]:
S = {k: json.load(open(ROOT/'reports'/f'summary_{k}.json')) for k in ['bike','pm25']}
cols = ['rows','n_series','freq','start','end','missing_steps','dup_ts','n_na','target_mean','target_cv','target_skew','target_kurt','target_outlier_pct','trend_strength','seasonal_strength']
tab = pd.DataFrame(S).T[cols]
tab.insert(0, 'license', ['CC BY 4.0','CC BY 4.0'])
tab.insert(1, 'mục tiêu', ['cnt','pm2.5'])
tab.insert(2, 'ngoại sinh', ['thời tiết, lịch','khí tượng, hướng gió'])
display(tab)
for k in S: print(k, 'ACF:', S[k]['acf'], '| dừng:', S[k]['stationarity'])

,license,mục tiêu,ngoại sinh,rows,n_series,freq,start,end,missing_steps,dup_ts,n_na,target_mean,target_cv,target_skew,target_kurt,target_outlier_pct,trend_strength,seasonal_strength
bike,CC BY 4.0,cnt,"thời tiết, lịch",17379,1,giờ,2011-01-01 00:00:00,2012-12-31 23:00:00,165,0,0,189.463,0.957,1.277,1.417,2.906,0.889,0.881
pm25,CC BY 4.0,pm2.5,"khí tượng, hướng gió",43824,1,giờ,2010-01-01 00:00:00,2014-12-31 23:00:00,0,0,2067,98.613,0.933,1.802,4.769,4.246,0,0.411


bike ACF: {'lag1': 0.843, 'lag2': 0.592, 'lag24': 0.819, 'lag168': 0.876} | dừng: {'cnt giờ': 'trái chiều: ADF nói dừng, KPSS nói không dừng (thường do xu hướng/mùa vụ → kiểm lại sau sai phân)', 'cnt ngày': 'không dừng', 'cnt ngày, sai phân 1': 'dừng'}
pm25 ACF: {'lag1': 0.965, 'lag2': 0.92, 'lag6': 0.747, 'lag24': 0.398, 'lag168': 0.023} | dừng: {'pm25 giờ (bỏ NA)': 'dừng', 'pm25 TB ngày': 'dừng'}


## Gợi ý tiền xử lý (dựa trên kết quả ở trên)

| Bộ | Gợi ý |
|---|---|
| **Bike** | Bỏ `casual`, `registered` (rò rỉ mục tiêu), `instant`; giữ một trong `temp`/`atemp`. Dựng lưới giờ đều, quyết định điền giờ thiếu (0 hoặc nội suy) và đánh dấu bằng cờ. Biến đổi log1p mục tiêu (lệch phải). Chia train/test theo thời gian; chú ý tăng trưởng giữa 2011→2012 (drift) — cân nhắc đặc trưng xu hướng hoặc chuẩn hoá theo cửa sổ. Đặc trưng: giờ, thứ, workingday, lag 24/168. |
| **PM2.5** | Mục tiêu thiếu theo đoạn: nội suy đoạn ngắn, mask đoạn dài khi tính loss/metric; bỏ 24 giờ đầu toàn NA. log1p mục tiêu (lệch phải nặng). One-hot `cbwd`; `Iws/Is/Ir` là cộng dồn → cân nhắc sai phân. Đỉnh ô nhiễm là tín hiệu thật, không xoá outlier. |
| **Chung** | Chia theo thời gian (không xáo trộn), fit scaler/imputer chỉ trên train, luôn báo cáo baseline naive và seasonal naive. |
